# Memory Selectivity Analysis — Counterfact (5 metrics)

Replays `memory_analysis.ipynb`'s cell-21 analysis on counterfact, but computes
all 5 alignment metrics from `fig6_memory_selectivity_personamem.ipynb`. Goal: confirm
the counterfact result holds under M4 (`cos_diff_last`) — the metric we want to
use uniformly across the paper.

Counterfact has single-token answers, so M4 ≡ M2 (`cos_diff_mean`) and M5 ≡ M3
(`two_cos_mean`) by construction — but we plot all 5 to verify.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))


In [ ]:
import os
# Pin to GPU 7. GPU 6 is currently held by the personalization notebook's kernel
# (~58 GB), so loading another 8B model there would OOM. GPU 7 has ~33 GB free,
# enough for one 8B-chat model with the model-on-CPU swap between gnm and ablation.
# MUST be set before `import torch`.
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import random
import json
import re
from collections import defaultdict
import numpy as np
import torch
import torch.nn.functional as F
from gnm import MemoryLLMChatWrapper, build_model
from config_memoryllm_train import (
    MemoryllmTrainConfig, DistributedCfg, DataCfg, SamplingCfg,
    ModelCfg, LoopCtrlCfg, LoggingCfg,
)
import matplotlib.pyplot as plt
from scipy import stats

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.device_count() =", torch.cuda.device_count())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0) =", torch.cuda.get_device_name(0))


# Helpers (verbatim copy from `memory_analysis.ipynb` cell 2)

In [ ]:
def find_answer_token_positions(wrapper, text: str, answer: str):
    """
    Find the token position(s) of the answer within a tokenized text.
    Returns list of (start_idx, end_idx) tuples for each occurrence.
    """
    # Tokenize the full text
    text_tokens = wrapper.tokenizer(text, return_tensors="pt", add_special_tokens=False)
    text_ids = text_tokens.input_ids[0].tolist()
    
    # Tokenize just the answer
    answer_tokens = wrapper.tokenizer(answer, return_tensors="pt", add_special_tokens=False)
    answer_ids = answer_tokens.input_ids[0].tolist()
    
    # Also try with leading space (common tokenization pattern)
    answer_with_space = " " + answer
    answer_space_tokens = wrapper.tokenizer(answer_with_space, return_tensors="pt", add_special_tokens=False)
    answer_space_ids = answer_space_tokens.input_ids[0].tolist()
    
    positions = []
    
    # Search for answer tokens in text
    for search_ids in [answer_ids, answer_space_ids]:
        for i in range(len(text_ids) - len(search_ids) + 1):
            if text_ids[i:i+len(search_ids)] == search_ids:
                positions.append((i, i + len(search_ids)))
    
    # Remove duplicates while preserving order
    seen = set()
    unique_positions = []
    for pos in positions:
        if pos not in seen:
            seen.add(pos)
            unique_positions.append(pos)
    
    return unique_positions, text_ids, answer_ids



@torch.no_grad()
def get_memory_delta_per_layer(
    wrapper: MemoryLLMChatWrapper,
    learning_instruction: str,
    document: str,
):
    """
    Returns memory delta for each layer.
    Output shape: [L, num_memory_tokens, hidden_dim]
    """
    wrapper.reset_memory()
    wrapper.model.eval()

    memory_context = (
        f"<|learning_instruction_start|>{learning_instruction}<|learning_instruction_end|>"
        f"<|document_start|>{document}<|document_end|>"
    )

    ctx_ids = wrapper.tokenizer(
        memory_context,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.to(wrapper.model.device)

    # Get memory delta without committing
    delta = wrapper.model.inject_memory(ctx_ids, update_memory=False)
    
    # delta is [1, L, num_memory_tokens, hidden_dim]
    return delta[0].detach()  # [L, num_memory_tokens, hidden_dim]



@torch.no_grad()
def get_hidden_states_during_injection(wrapper, learning_instruction: str, document: str):
    """
    Get hidden states at ALL layers during the memory INJECTION forward pass.
    
    IMPORTANT: We call wrapper.model(...) with is_injection=True to get the 
    memory-influenced hidden states, NOT wrapper.model.model(...) which would
    bypass the memory machinery entirely.
    
    Returns: dict with 'hidden_states' tensor of shape [L+1, seq_len, D]
             where seq_len matches the input_ids length (memory tokens are internal)
    """
    wrapper.reset_memory()
    wrapper.model.eval()
    
    memory_context = (
        f"<|learning_instruction_start|>{learning_instruction}<|learning_instruction_end|>"
        f"<|document_start|>{document}<|document_end|>"
    )
    
    ctx_ids = wrapper.tokenizer(
        memory_context,
        return_tensors="pt",
        add_special_tokens=False
    ).input_ids.to(wrapper.model.device)
    
    # CRITICAL: Call the MemoryLLM forward with is_injection=True
    # This runs the actual memory injection forward pass where memory tokens
    # are prepended at each layer. The returned hidden_states correspond to
    # the input_ids positions (memory influence is internal).
    outputs = wrapper.model(
        input_ids=ctx_ids,
        output_hidden_states=True,
        is_injection=True,  # Memory write mode
        return_dict=True
    )
    
    # hidden_states is tuple of (L+1) tensors, each [1, seq_len, D]
    # These are the hidden states for the INPUT tokens (not memory tokens)
    # but they have been computed WITH memory tokens present in attention
    hidden_states = torch.stack([h.squeeze(0) for h in outputs.hidden_states], dim=0)
    
    return {
        'hidden_states': hidden_states.detach(),  # [L+1, seq_len, D]
        'input_ids': ctx_ids.squeeze(0).detach(),  # [seq_len]
        'memory_context': memory_context,  # for debugging
    }


# Model loading: GNM (counterfact-trained) + Ablation (untrained MemoryLLM, matching the original `memory_analysis.ipynb`)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = MemoryllmTrainConfig(
    run_name    = "run",
    seed        = 42,
    distributed = DistributedCfg(cuda_visible_devices="0", nproc_per_node=4, master_port=29512),
    data        = DataCfg(),
    sampling    = SamplingCfg(),
    opt         = None,
    model       = ModelCfg(model='memoryllm_chat'),
    loop        = LoopCtrlCfg(),
    log         = LoggingCfg(),
)

print("Loading GNM (counterfact-trained)...")
gnm_model_wrapper = build_model(cfg)
GNM_PATH = "../models/gnm_counterfact/model.pth"
gnm_state_dict = torch.load(GNM_PATH, map_location='cpu')
gnm_model_wrapper.load_state_dict(gnm_state_dict)
gnm_model_wrapper.eval()
gnm_model_wrapper.to(device)
print("  done")

print("Loading Ablation (untrained MemoryLLM-8B-chat per original notebook)...")
ablation_model_wrapper = build_model(cfg)
# To use the trained ablation variant instead, uncomment:
ABL_PATH = "../models/gnm_counterfact_ablation/model.pth"
ablation_state_dict = torch.load(ABL_PATH, map_location='cpu')
ablation_model_wrapper.load_state_dict(ablation_state_dict)
ablation_model_wrapper.eval()
# Keep ablation on CPU while gnm runs — GPU 7 has ~33 GB free, only enough for one
# 8B model at a time. Will .to(device) right before the ablation collect pass.
ablation_model_wrapper.to('cpu')
torch.cuda.empty_cache()
print("  done (kept on CPU)")


# Dataset (counterfact write-invariance, same file as the original notebook)

In [ ]:
DATASET_PATH = "../data/analysis/counterfact_memory_analysis.json"
with open(DATASET_PATH) as f:
    dataset = json.load(f)
print(f"loaded {len(dataset)} target-fact sets, total examples = {sum(len(e['examples']) for e in dataset)}")
print("first set keys:", list(dataset[0].keys()))
print("first example keys:", list(dataset[0]['examples'][0].keys()))


# Sanity check: verify token spans for first 2 examples

In [ ]:
for i in range(2):
    entry = dataset[0]
    ex = entry['examples'][i]
    target_answer = entry['target_fact']['requested_rewrite']['target_new']['str']
    distractor_answer = ex['distractor_fact']['requested_rewrite']['target_new']['str']
    print(f"\n--- Example {i} ---")
    print(f"target_answer:     {target_answer!r}")
    print(f"distractor_answer: {distractor_answer!r}")
    memory_context = (
        f"<|learning_instruction_start|>{entry['learning_instruction']}<|learning_instruction_end|>"
        f"<|document_start|>{ex['document']}<|document_end|>"
    )
    t_pos, ctx_ids, _ = find_answer_token_positions(gnm_model_wrapper, memory_context, target_answer)
    d_pos, _, _ = find_answer_token_positions(gnm_model_wrapper, memory_context, distractor_answer)
    print(f"target positions (first 3):     {t_pos[:3]}")
    print(f"distractor positions (first 3): {d_pos[:3]}")
    if t_pos:
        s, ed = t_pos[0]
        print(f"target decoded:     {gnm_model_wrapper.tokenizer.decode(ctx_ids[s:ed])!r}")
    if d_pos:
        s, ed = d_pos[0]
        print(f"distractor decoded: {gnm_model_wrapper.tokenizer.decode(ctx_ids[s:ed])!r}")


# 5-metric `compute_memory_selectivity` (identical to personalization notebook)

In [ ]:
METRIC_NAMES = ["dot", "cos_diff_mean", "two_cos_mean", "cos_diff_last", "two_cos_last"]
METRIC_DESC = {
    "dot":           r"$\langle d,\;\widehat{(\bar{t}-\bar{r})}\rangle$  (cell-21 original)",
    "cos_diff_mean": r"$\langle\hat{d},\;\widehat{(\bar{t}-\bar{r})}\rangle$  (cosine of mean-pooled difference)",
    "two_cos_mean":  r"$\langle\hat{d},\hat{\bar{t}}\rangle-\langle\hat{d},\hat{\bar{r}}\rangle$  (Option 1: two cosines, mean-pooled)",
    "cos_diff_last": r"$\langle\hat{d},\;\widehat{(t_{\text{last}}-r_{\text{last}})}\rangle$  (Option 2: last-token, cosine of difference)",
    "two_cos_last":  r"$\langle\hat{d},\hat{t}_{\text{last}}\rangle-\langle\hat{d},\hat{r}_{\text{last}}\rangle$  (Option 3: last-token, two cosines)",
}


def _unit(v):
    return F.normalize(v.unsqueeze(0), dim=1).squeeze(0)


@torch.no_grad()
def compute_memory_selectivity(wrapper, learning_instruction: str, document: str,
                                target_answer: str, distractor_answer: str):
    """One forward pass; five alignment metrics per layer."""
    memory_context = (
        f"<|learning_instruction_start|>{learning_instruction}<|learning_instruction_end|>"
        f"<|document_start|>{document}<|document_end|>"
    )
    target_positions, ctx_ids, _ = find_answer_token_positions(wrapper, memory_context, target_answer)
    distractor_positions, _, _ = find_answer_token_positions(wrapper, memory_context, distractor_answer)
    if not target_positions or not distractor_positions:
        return None, "Answer not found"

    delta = get_memory_delta_per_layer(wrapper, learning_instruction, document)  # [L, 256, D]
    result = get_hidden_states_during_injection(wrapper, learning_instruction, document)
    hidden_states = result["hidden_states"]  # [L+1, seq_len, D]

    target_start, target_end = target_positions[0]
    distractor_start, distractor_end = distractor_positions[0]

    L = delta.shape[0]
    out = {k: [] for k in METRIC_NAMES}
    for layer in range(L - 1):
        h_idx = layer + 1
        t_span = hidden_states[h_idx, target_start:target_end, :].float()
        r_span = hidden_states[h_idx, distractor_start:distractor_end, :].float()
        t_mean = t_span.mean(dim=0)
        r_mean = r_span.mean(dim=0)
        t_last = t_span[-1]
        r_last = r_span[-1]

        d = delta[layer].float().mean(dim=0).to(t_mean.device)
        d_hat = _unit(d)

        v_mean_hat = _unit(t_mean - r_mean)
        out["dot"].append((d @ v_mean_hat).item())
        out["cos_diff_mean"].append((d_hat @ v_mean_hat).item())
        out["two_cos_mean"].append(
            (d_hat @ _unit(t_mean)).item() - (d_hat @ _unit(r_mean)).item()
        )

        v_last_hat = _unit(t_last - r_last)
        out["cos_diff_last"].append((d_hat @ v_last_hat).item())
        out["two_cos_last"].append(
            (d_hat @ _unit(t_last)).item() - (d_hat @ _unit(r_last)).item()
        )

    return {k: np.array(v) for k, v in out.items()}, None


# Collect across the 200 (target_fact, example) pairs

In [ ]:
def collect_counterfact(wrapper, dataset, log_every=20):
    per_metric = {k: [] for k in METRIC_NAMES}
    errors = []
    count = 0
    total = sum(len(entry['examples']) for entry in dataset)
    for entry in dataset:
        learning_instruction = entry['learning_instruction']
        target_answer = entry['target_fact']['requested_rewrite']['target_new']['str']
        for ex in entry['examples']:
            if count % log_every == 0:
                print(f"  {count}/{total}")
            distractor_answer = ex['distractor_fact']['requested_rewrite']['target_new']['str']
            out, err = compute_memory_selectivity(
                wrapper,
                learning_instruction=learning_instruction,
                document=ex['document'],
                target_answer=target_answer,
                distractor_answer=distractor_answer,
            )
            if err is not None:
                errors.append((count, err))
            else:
                for k in METRIC_NAMES:
                    per_metric[k].append(out[k])
            count += 1
            if count % 10 == 9:
                torch.cuda.empty_cache()
    arrs = {k: np.stack(v, axis=0) if v else None for k, v in per_metric.items()}
    return arrs, errors

print("\nCollecting for GNM (counterfact-trained)...")
gnm_metrics, gnm_errors = collect_counterfact(gnm_model_wrapper, dataset)
print(f"  collected={gnm_metrics['dot'].shape[0] if gnm_metrics['dot'] is not None else 0}  errors={len(gnm_errors)}")
torch.cuda.empty_cache()

# Free GNM from GPU and bring ablation onto GPU.
gnm_model_wrapper.to('cpu')
torch.cuda.empty_cache()
ablation_model_wrapper.to(device)

print("\nCollecting for Ablation...")
ablation_metrics, ablation_errors = collect_counterfact(ablation_model_wrapper, dataset)
print(f"  collected={ablation_metrics['dot'].shape[0] if ablation_metrics['dot'] is not None else 0}  errors={len(ablation_errors)}")
torch.cuda.empty_cache()


# Plot all five metrics side-by-side

Same plotting code as the personalization notebook, with GNM in blue and Ablation in red
to match the original counterfact figure.

In [ ]:
def per_layer_summary(arr):
    return arr.mean(axis=0), arr.std(axis=0) / np.sqrt(arr.shape[0])

summaries = {}
for k in METRIC_NAMES:
    a_mean, a_se = per_layer_summary(ablation_metrics[k])
    g_mean, g_se = per_layer_summary(gnm_metrics[k])
    summaries[k] = {'a_mean': a_mean, 'a_se': a_se, 'g_mean': g_mean, 'g_se': g_se}

num_layers = len(summaries['dot']['a_mean'])
layers = list(range(num_layers))
mid_layers = slice(15, -1)

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()
for i, k in enumerate(METRIC_NAMES):
    ax = axes[i]
    s = summaries[k]
    ax.plot(layers, s['a_mean'], 'r-s', label='Ablation', linewidth=2, markersize=3)
    ax.fill_between(layers, s['a_mean']-1.96*s['a_se'], s['a_mean']+1.96*s['a_se'], alpha=0.2, color='red')
    ax.plot(layers, s['g_mean'], 'b-o', label='GNM', linewidth=2, markersize=3)
    ax.fill_between(layers, s['g_mean']-1.96*s['g_se'], s['g_mean']+1.96*s['g_se'], alpha=0.2, color='blue')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Score')
    ax.set_title(f'M{i+1} — {k}\n' + METRIC_DESC[k], fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

# Final panel: mid-layer summary bar chart across all 5 metrics.
ax = axes[5]
x = np.arange(len(METRIC_NAMES))
width = 0.35
a_mids, a_mid_ses, g_mids, g_mid_ses = [], [], [], []
for k in METRIC_NAMES:
    a_arr = ablation_metrics[k][:, mid_layers].mean(axis=1)
    g_arr = gnm_metrics[k][:, mid_layers].mean(axis=1)
    a_mids.append(a_arr.mean());      g_mids.append(g_arr.mean())
    a_mid_ses.append(a_arr.std()/np.sqrt(len(a_arr)))
    g_mid_ses.append(g_arr.std()/np.sqrt(len(g_arr)))
ax.bar(x - width/2, a_mids, width, yerr=[1.96*s for s in a_mid_ses], label='Ablation', color='red', alpha=0.7, capsize=4)
ax.bar(x + width/2, g_mids, width, yerr=[1.96*s for s in g_mid_ses], label='GNM', color='blue', alpha=0.7, capsize=4)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(METRIC_NAMES, rotation=20, ha='right', fontsize=9)
ax.set_ylabel(f'Mean (layers {mid_layers.start}-{mid_layers.stop})')
ax.set_title(f'Mid-layer averages across metrics')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
PLOT_PATH = '../plots/memory_selectivity_counterfact_5metrics.png'
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved 5-metric plot to {PLOT_PATH}')

# Stats per metric (paired t-test, Cohen's d) on mid-layer averages.
print(f"\nMid-layer slice {mid_layers}: per-metric stats (GNM vs Ablation)")
print(f"{'metric':<18} {'a_mid':>8} {'g_mid':>8} {'t':>7} {'p':>9} {'d':>7}  {'pct_g>a':>8}")
stats_rows = {}
for k in METRIC_NAMES:
    a_arr = ablation_metrics[k][:, mid_layers].mean(axis=1)
    g_arr = gnm_metrics[k][:, mid_layers].mean(axis=1)
    t_stat, p_val = stats.ttest_rel(g_arr, a_arr)
    diff = g_arr - a_arr
    cd = float(diff.mean()/diff.std()) if diff.std() > 0 else 0.0
    pct = float((g_arr > a_arr).mean()*100)
    stats_rows[k] = {'a_mid': float(a_arr.mean()), 'g_mid': float(g_arr.mean()),
                      't': float(t_stat), 'p': float(p_val), 'd': cd, 'pct_g_higher': pct}
    print(f"{k:<18} {a_arr.mean():>8.4f} {g_arr.mean():>8.4f} {t_stat:>7.2f} {p_val:>9.4g} {cd:>7.3f}  {pct:>7.1f}%")

JSON_PATH = '../evals/counterfact/paper_results/memory_analysis/memory_selectivity_counterfact.json'
with open(JSON_PATH, 'w') as f:
    json.dump({
        'metrics': {
            k: {
                'ablation_mean_per_layer': summaries[k]['a_mean'].tolist(),
                'ablation_se_per_layer':   summaries[k]['a_se'].tolist(),
                'gnm_mean_per_layer':      summaries[k]['g_mean'].tolist(),
                'gnm_se_per_layer':        summaries[k]['g_se'].tolist(),
                **stats_rows[k],
            }
            for k in METRIC_NAMES
        },
        'metric_descriptions': METRIC_DESC,
        'mid_layer_slice':     [mid_layers.start, mid_layers.stop],
        'num_examples':        int(gnm_metrics['dot'].shape[0]),
        'num_gnm_errors':      len(gnm_errors),
        'num_ablation_errors': len(ablation_errors),
        'dataset_path':        DATASET_PATH,
        'gnm_checkpoint_path': GNM_PATH,
    }, f, indent=2)
print(f'Saved JSON to {JSON_PATH}')
